In [ ]:
# imports
%load_ext autoreload
%autoreload 2
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd
from astropy.cosmology import FlatLambdaCDM
import corner

# ad-hoc fix for imports 
import os
import sys
dirname = os.getcwd()
sys.path.insert(0, os.path.join(dirname, '../..'))
import tdc_sampler
from Utils.inference_utils import median_sigma_from_samples
from Utils.mcmc_utils import median_and_uncertainty, DE_fom
from Utils.tdc_utils import ddt_from_redshifts, td_from_ddt_fpd
sys.path.insert(0, os.path.join(dirname, '../../../sbi-stronglensing'))
from src.analysis.visualization_utils import plot_corner, plot_corner_overlay, plot_corner_image, plot_parity, table_of_metrics, plot_coverage
from src.analysis.diagnostic_utils import get_coverage

# moved helper functions to script
from completo_h0test_utils import dv_dict_from_completo, truth_and_posteriors, dv_dict_from_completo_extendedCPDF

In [ ]:
GROUNDTRUTH_COSMO = FlatLambdaCDM(H0=70.,Om0=0.3)
MDN_COLOR = 'slateblue'
NSF_COLOR = 'mediumseagreen'

In [ ]:
new_nsf_dir = '/Users/smericks/Desktop/StrongLensing/project3/sbi-stronglensing/nsf/nsf_resnet34_2026-07-05_08-37-20/'
dv_dict_dbls_nsf, dv_dict_quads_nsf = dv_dict_from_completo(
    (new_nsf_dir+'1ktest_seed7_5ksamps.npy'),
    (new_nsf_dir+'test10k_seed7.h5'),
    chosen_idxs=narrow_gamma_tenplus_td, GROUNDTRUTH_COSMO=GROUNDTRUTH_COSMO)

#dv_dict_dbls_mdn, dv_dict_quads_mdn = dv_dict_from_completo('DataVectors/completo/mdn_1ktest_seed7_5ksamps.npy','DataVectors/completo/10ktest_seed7.h5',
#    chosen_idxs=narrow_gamma_tenplus_td, GROUNDTRUTH_COSMO=GROUNDTRUTH_COSMO)

## Testing the full cov. cPDF framework ##

In [ ]:
my_chain_nsf_cPDF = tdc_sampler.fast_TDC([lklhd_obj_quads,lklhd_obj_dbls], 
    [dv_dict_quads_nsf_CPDF,dv_dict_dbls_nsf_CPDF], num_emcee_samps=200,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee',num_cpdf_params=2) # NOTE: we have to specify this now!!

In [ ]:
# Verify things ran correctly by looking at the chains

def plot_convergence_by_walker(samples_mcmc, param_mcmc, verbose = False):
    n_params = samples_mcmc.shape[2]
    n_step = int(samples_mcmc.shape[1])
    chain = samples_mcmc
    mean_pos = np.zeros((n_params, n_step))
    median_pos = np.zeros((n_params, n_step))
    std_pos = np.zeros((n_params, n_step))
    q16_pos = np.zeros((n_params, n_step))
    q84_pos = np.zeros((n_params, n_step))
    # chain = np.empty((nwalker, nstep, ndim), dtype = np.double)
    for i in np.arange(n_params):
        for j in np.arange(n_step):
            mean_pos[i][j] = np.mean(chain[:, j, i])
            median_pos[i][j] = np.median(chain[:, j, i])
            std_pos[i][j] = np.std(chain[:, j, i])
            q16_pos[i][j] = np.percentile(chain[:, j, i], 16.)
            q84_pos[i][j] = np.percentile(chain[:, j, i], 84.)
    fig, ax = plt.subplots(n_params, sharex=True, figsize=(16, 2 * n_params))
    if n_params == 1: ax = [ax]
    last = n_step
    burnin = int((9.*n_step) / 10.) #get the final value on the last 10% on the chain
    for i in range(n_params):
        if verbose :
            print(param_mcmc[i], '{:.4f} +/- {:.4f}'.format(median_pos[i][last - 1], (q84_pos[i][last - 1] - q16_pos[i][last - 1]) / 2))
        ax[i].plot(median_pos[i][:last], c='g')
        ax[i].axhline(np.median(median_pos[i][burnin:last]), c='r', lw=1)
        ax[i].fill_between(np.arange(last), q84_pos[i][:last], q16_pos[i][:last], alpha=0.4)
        ax[i].set_ylabel(param_mcmc[i], fontsize=10)
        ax[i].set_xlim(0, last)
    return fig

#with h5py.File('DataVectors/gold/baseline_chain.h5','r') as h5:
#    test_chain = h5['mcmc_chain'][:]
import emcee
reader = emcee.backends.HDFBackend('InferenceRuns/with_lenscenter_200lenses_CPDF_backend.h5', read_only=True)
test_chain = reader.get_chain()
my_chain_nsf_cPDF = test_chain
plot_convergence_by_walker(np.transpose(test_chain,axes=(1,0,2)),
    ['$H_0$','$\Omega_M$',
     r'$\mu(log($\theta_E$)','$\mu(log(\gamma_{lens})))$','chol1','chol2','chol3'])

In [ ]:
def unpack_fullcPDF_chain(emcee_chain,burnin,mu_lp,stddev_lp):

    # condense walkers dim and remove burnin
    print('emcee_chain shape: ', emcee_chain.shape)
    my_chain = emcee_chain[burnin:].reshape((-1,emcee_chain.shape[2]))
    print('my_chain shape', my_chain.shape)

    num_cp = 2
    # here comes the lens params cPDF
    num_h = np.shape(my_chain)[-1] - num_cp # num_h = N + N(N+1)/2 (num. hyperparameters)
    # solv quad. eqn. to determine # of lens params (0 = n^2 + 3n -2(num_h))
    num_lp = int((-3 + np.sqrt(9+8*num_h))/2) # we only need the positive solution...
    
    # Extract parameters
    means_samples = my_chain[:, num_cp:(num_lp + num_cp)]
    chol_samples = my_chain[:, (num_lp + num_cp):]
    
    n_samples = chol_samples.shape[0]
    
    # Vectorized L matrix construction
    L_matrices = np.zeros((n_samples, num_lp, num_lp))
    tril_indices = np.tril_indices(num_lp)
    L_matrices[:, tril_indices[0], tril_indices[1]] = chol_samples
    
    # Compute all covariances at once using einsum
    cov_matrices = np.einsum('nij,nkj->nik', L_matrices, L_matrices)

    # use mu_lp and stddev_lp to undo the normalization to mu=0,sigma=1
    # Means: simple rescaling
    means_original = means_samples * stddev_lp + mu_lp  # shape: (n_samples, num_lp)
    
    # Covariances: need to scale by outer product of stddevs
    # Cov_original = diag(sigma) @ Cov_normalized @ diag(sigma)
    stddev_matrix = np.diag(stddev_lp)  # shape: (num_lp, num_lp)
    cov_original = np.einsum('ij,njk,kl->nil', stddev_matrix, cov_matrices, stddev_matrix)

    # return samples of cosmo parameters, samples of means, and samples of full cov. matrices...
    return my_chain[:,:num_cp], means_original, cov_original

In [ ]:
# understanding ground truth population
cPDF_param_labels = [
        'deflector_LOG_theta_E','deflector_LOG_gamma_pl'
    ]
from src.training.data_loader import load_hdf5_labels
theta_training = np.asarray(load_hdf5_labels(
    '/Users/smericks/Desktop/StrongLensing/project3/slsim/slsim/TrainingSets/100ktraining_seed1.h5',
    cPDF_param_labels))

# training distribution
figure = corner.corner(theta_training[:5000],plot_datapoints=False,
        color='gray',levels=[0.68,0.95],fill_contours=True,
        labels= ['log(theta_E)','log(gamma_pl)'],
        dpi=300,hist_kwargs={'density':True},
        fig=None,label_kwargs={'fontsize':20},smooth=2)

# test set distribution
(truth_vals_dict_nsf, dbls_idxs, quads_idxs, 
fasttdc_fpd_samps_dbls_nsf, fasttdc_fpd_samps_quads_nsf) = truth_and_posteriors(
    'DataVectors/completo/nsf_samps_1k_seed7.npy',
    'DataVectors/completo/10ktest_seed7.h5',
    chosen_idxs=narrow_gamma_tenplus_td, GROUNDTRUTH_COSMO=GROUNDTRUTH_COSMO)

test_set_samps = np.stack((truth_vals_dict_nsf['deflector_LOG_theta_E'],
    truth_vals_dict_nsf['deflector_LOG_gamma_pl']),axis=1)

corner.corner(test_set_samps,plot_datapoints=False,
        color='maroon',levels=[0.68,0.95],fill_contours=True,
        labels= ['log(theta_E)','log(gamma_pl)'],
        dpi=300,hist_kwargs={'density':True},
        fig=figure,label_kwargs={'fontsize':20},smooth=2)

# fit a Gaussian to these samps
# fit a Gaussian to these samps
mean_test_set = np.mean(test_set_samps, axis=0)
cov_test_set = np.cov(test_set_samps.T)

# unpack from chain
cosmo_samps, mu_samps, cov_mat_samps = unpack_fullcPDF_chain(my_chain_nsf_cPDF,40,
    mu_lp=dv_dict_dbls_nsf_CPDF['mu_norm_cPDF_params'],
    stddev_lp=dv_dict_dbls_nsf_CPDF['stddev_norm_cPDF_params'])
median_mean = np.median(mu_samps,axis=0)
median_cov = np.median(cov_mat_samps,axis=0)
from scipy.stats import multivariate_normal
median_samps = multivariate_normal.rvs(mean=median_mean,cov=median_cov,size=5000)
corner.corner(median_samps,plot_datapoints=False,
        color=NSF_COLOR,levels=[0.68,0.95],fill_contours=True,
        labels= [r'log $(\theta_E)$',r'log $(\gamma_{lens})$'],
        dpi=300,hist_kwargs={'density':True},
        fig=figure,label_kwargs={'fontsize':20},smooth=2)


custom_lines = [Line2D([0], [0], color='grey', lw=10),
                Line2D([0], [0], color='maroon', lw=10),
                Line2D([0], [0], color=NSF_COLOR, lw=10)]
custom_labels = ['Prior','Test Set','Inferred (NSF)']


axes = np.array(figure.axes).reshape((2, 2))
axes[0,2-1].legend(custom_lines,custom_labels,frameon=False,fontsize=13)


In [ ]:
# understanding ground truth population
cPDF_param_labels = [
        'deflector_LOG_theta_E','deflector_LOG_gamma_pl'
    ]
from src.training.data_loader import load_hdf5_labels
theta_training = np.asarray(load_hdf5_labels(
    '/Users/smericks/Desktop/StrongLensing/project3/slsim/slsim/TrainingSets/100ktraining_seed1.h5',
    cPDF_param_labels))

# training distribution
figure = corner.corner(np.exp(theta_training[:5000]),plot_datapoints=False,
        color='gray',levels=[0.68,0.95],fill_contours=True,
        labels= ['log(theta_E)','log(gamma_pl)'],
        dpi=300,hist_kwargs={'density':True},
        fig=None,label_kwargs={'fontsize':20},smooth=2)

# test set distribution
(truth_vals_dict_nsf, dbls_idxs, quads_idxs, 
fasttdc_fpd_samps_dbls_nsf, fasttdc_fpd_samps_quads_nsf) = truth_and_posteriors(
    'DataVectors/completo/nsf_samps_1k_seed7.npy',
    'DataVectors/completo/10ktest_seed7.h5',
    chosen_idxs=narrow_gamma_tenplus_td, GROUNDTRUTH_COSMO=GROUNDTRUTH_COSMO)

test_set_samps = np.stack((truth_vals_dict_nsf['deflector_LOG_theta_E'],
    truth_vals_dict_nsf['deflector_LOG_gamma_pl']),axis=1)

corner.corner(np.exp(test_set_samps),plot_datapoints=False,
        color='maroon',levels=[0.68,0.95],fill_contours=True,
        labels= ['log(theta_E)','log(gamma_pl)'],
        dpi=300,hist_kwargs={'density':True},
        fig=figure,label_kwargs={'fontsize':20},smooth=2)

# fit a Gaussian to these samps
# fit a Gaussian to these samps
mean_test_set = np.mean(test_set_samps, axis=0)
cov_test_set = np.cov(test_set_samps.T)

# unpack from chain
cosmo_samps, mu_samps, cov_mat_samps = unpack_fullcPDF_chain(my_chain_nsf_cPDF,40,
    mu_lp=dv_dict_dbls_nsf_CPDF['mu_norm_cPDF_params'],
    stddev_lp=dv_dict_dbls_nsf_CPDF['stddev_norm_cPDF_params'])
median_mean = np.median(mu_samps,axis=0)
median_cov = np.median(cov_mat_samps,axis=0)
from scipy.stats import multivariate_normal
median_samps = multivariate_normal.rvs(mean=median_mean,cov=median_cov,size=5000)
corner.corner(np.exp(median_samps),plot_datapoints=False,
        color=NSF_COLOR,levels=[0.68,0.95],fill_contours=True,
        labels= [r'$\theta_E$',r'$\gamma_{lens}$'],
        dpi=300,hist_kwargs={'density':True},
        fig=figure,label_kwargs={'fontsize':20},smooth=2)


custom_lines = [Line2D([0], [0], color='grey', lw=10),
                Line2D([0], [0], color='maroon', lw=10),
                Line2D([0], [0], color=NSF_COLOR, lw=10)]
custom_labels = ['Prior','Test Set','Inferred (NSF)']


axes = np.array(figure.axes).reshape((2, 2))
axes[0,2-1].legend(custom_lines,custom_labels,frameon=False,fontsize=13)

In [ ]:
# unpack from chain
cosmo_samps, mu_samps, cov_mat_samps = unpack_fullcPDF_chain(my_chain_nsf_cPDF,200,
    mu_lp=dv_dict_dbls_nsf_CPDF['mu_norm_cPDF_params'],
    stddev_lp=dv_dict_dbls_nsf_CPDF['stddev_norm_cPDF_params'])


corner_samps = np.stack((cosmo_samps[:,0],cosmo_samps[:,1],
    mu_samps[:,0],mu_samps[:,1],
    cov_mat_samps[:,0,0],cov_mat_samps[:,1,1],cov_mat_samps[:,0,1]),axis=1)

print(corner_samps.shape)

corner_samp_labels = ['$H_0$',r'$\Omega_{\text{m}}$',
        r'$\mu_{\text{log}(\theta_{E})}$',r'$\mu_{\text{log}(\gamma_{lens})}$',
        r'$\sigma_{log(\theta_{E})}^2$',r'$\sigma_{log(\gamma_{lens})}^2$',
        r'$\sigma_{log(\theta_{E})} \sigma_{log(\gamma_{lens})}$']

print(len(corner_samp_labels))
truths = [70.,0.3,mean_test_set[0],mean_test_set[1],cov_test_set[0,0],cov_test_set[1,1],cov_test_set[0,1]]

my_color = NSF_COLOR
figure = corner.corner(corner_samps,plot_datapoints=False,
    color=my_color,levels=[0.68,0.95],fill_contours=True,
    labels=corner_samp_labels,
    dpi=300,truths=truths,truth_color='black',
    fig=None,label_kwargs={'fontsize':20},smooth=2)


## TODO: feed into an inference now and see what happens!! ##

In [ ]:
# set up a likelihood object
lklhd_obj_dbls = tdc_sampler.TDCLikelihood(
    fpd_sample_shape=np.shape(dv_dict_dbls_nsf['fpd_samples']), # pre-define input shape for fermat potential differences
    cosmo_model='LCDM', # infers four params: [H0,OmegaM,mu(gamma),sigma(gamma)]
    use_astropy=True, # relic option, we always use astropy right now
    use_gamma_info=True) # whether to infer a population over gamma_lens or not

lklhd_obj_quads = tdc_sampler.TDCLikelihood(
    fpd_sample_shape=np.shape(dv_dict_quads_nsf['fpd_samples']), # pre-define input shape for fermat potential differences
    cosmo_model='LCDM', # infers four params: [H0,OmegaM,mu(gamma),sigma(gamma)]
    use_astropy=True, # relic option, we always use astropy right now
    use_gamma_info=True) # whether to infer a population over gamma_lens or not

In [ ]:
my_chain_nsf_dbls = tdc_sampler.fast_TDC([lklhd_obj_dbls], 
    [dv_dict_dbls_nsf], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
my_chain_nsf_quads = tdc_sampler.fast_TDC([lklhd_obj_quads], 
    [dv_dict_quads_nsf], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
my_chain_nsf_both = tdc_sampler.fast_TDC([lklhd_obj_quads,lklhd_obj_dbls], 
    [dv_dict_quads_nsf,dv_dict_dbls_nsf], num_emcee_samps=200,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
my_chain_mdn = tdc_sampler.fast_TDC([lklhd_obj_dbls], 
    [dv_dict_dbls_mdn], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
my_chain_mdn_quads = tdc_sampler.fast_TDC([lklhd_obj_quads], 
    [dv_dict_quads_mdn], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
my_chain_mdn_both = tdc_sampler.fast_TDC([lklhd_obj_quads,lklhd_obj_dbls], 
    [dv_dict_quads_mdn,dv_dict_dbls_mdn], num_emcee_samps=1000,
    n_walkers=20, use_mpi=False, use_multiprocess=True, backend_path=None, 
    reset_backend=True,sampler_type='emcee')

In [ ]:
# Verify things ran correctly by looking at the chains

def plot_convergence_by_walker(samples_mcmc, param_mcmc, verbose = False):
    n_params = samples_mcmc.shape[2]
    n_step = int(samples_mcmc.shape[1])
    chain = samples_mcmc
    mean_pos = np.zeros((n_params, n_step))
    median_pos = np.zeros((n_params, n_step))
    std_pos = np.zeros((n_params, n_step))
    q16_pos = np.zeros((n_params, n_step))
    q84_pos = np.zeros((n_params, n_step))
    # chain = np.empty((nwalker, nstep, ndim), dtype = np.double)
    for i in np.arange(n_params):
        for j in np.arange(n_step):
            mean_pos[i][j] = np.mean(chain[:, j, i])
            median_pos[i][j] = np.median(chain[:, j, i])
            std_pos[i][j] = np.std(chain[:, j, i])
            q16_pos[i][j] = np.percentile(chain[:, j, i], 16.)
            q84_pos[i][j] = np.percentile(chain[:, j, i], 84.)
    fig, ax = plt.subplots(n_params, sharex=True, figsize=(16, 2 * n_params))
    if n_params == 1: ax = [ax]
    last = n_step
    burnin = int((9.*n_step) / 10.) #get the final value on the last 10% on the chain
    for i in range(n_params):
        if verbose :
            print(param_mcmc[i], '{:.4f} +/- {:.4f}'.format(median_pos[i][last - 1], (q84_pos[i][last - 1] - q16_pos[i][last - 1]) / 2))
        ax[i].plot(median_pos[i][:last], c='g')
        ax[i].axhline(np.median(median_pos[i][burnin:last]), c='r', lw=1)
        ax[i].fill_between(np.arange(last), q84_pos[i][:last], q16_pos[i][:last], alpha=0.4)
        ax[i].set_ylabel(param_mcmc[i], fontsize=10)
        ax[i].set_xlim(0, last)
    return fig

import emcee
reader = emcee.backends.HDFBackend('InferenceRuns/mdn/with_lenscenter_200lenses_backend.h5', read_only=True)
test_chain = reader.get_chain()
plot_convergence_by_walker(np.transpose(test_chain,axes=(1,0,2)),
    ['$H_0$','$\Omega_M$',
     r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'])

In [ ]:
n_samps = 500
n_walkers = 20
param_idx = 0
for i in range(0,n_walkers):
    plt.scatter(range(0,n_samps),test_chain[:,i,param_idx],s=2)
plt.title('$H_0$ Chains (NSF)')

In [ ]:
import emcee
reader = emcee.backends.HDFBackend('InferenceRuns/mdn/with_lenscenter_200lenses_backend.h5', read_only=True)
test_chain_mdn = reader.get_chain()

reader = emcee.backends.HDFBackend('InferenceRuns/nsf/with_lenscenter_200lenses_backend.h5', read_only=True)
test_chain_nsf = reader.get_chain()

In [ ]:
# Corner plots!

H0_TRUTH = 70.
OMEGAM_TRUTH = 0.3
MEAN_GAMMA_TRUTH = 2.03
SIGMA_GAMMA_TRUTH = 0.09

from scipy.stats import norm

exp_chains = [
    np.transpose(test_chain_mdn,axes=(1,0,2)),
    np.transpose(test_chain_nsf,axes=(1,0,2)),
    ]
    #np.transpose(my_chain_both,axes=(1,0,2))]
exp_names = ['MDN Modeling', 'NSF Modeling']

num_chains = len(exp_chains)
burnin = [200,200,200]
cmap = plt.get_cmap('ocean')
colors = [MDN_COLOR, NSF_COLOR] #goldenrod',
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    median_and_uncertainty(exp_chain,burnin[i])
    #zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$',r'$\Omega_{\text{m}}$',
                r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'],
            dpi=300,truths=[H0_TRUTH,OMEGAM_TRUTH,MEAN_GAMMA_TRUTH,SIGMA_GAMMA_TRUTH],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':40},smooth=2)

    else:

        corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$',r'$\Omega_{\text{m}}$',
                r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'],
            dpi=300,#truths=[H0_TRUTH,OMEGAM_TRUTH,MEAN_GAMMA_TRUTH,SIGMA_GAMMA_TRUTH],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':40},smooth=2)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=10))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.1f$\pm$%.2f \n $\Omega_m$=%.3f$\pm$%.3f'%(
        np.round(h0,decimals=2), np.round(h0_sigma,decimals=2), 
        np.round(OmegaM,decimals=3), np.round(OmegaM_sigma,decimals=3)))

axes = np.array(figure.axes).reshape((4, 4))
axes[0,num_params-1].legend(custom_lines,custom_labels,frameon=False,fontsize=20)


bounds = [[67,74],[0.05,0.5],[2.,2.13],[0.05,0.13]]
for r in range(0,4):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

#plt.tight_layout()
plt.savefig('/Users/smericks/Desktop/StrongLensing/project3/Figures/lcdm_cosmo.pdf',bbox_inches='tight')


## Debugging what's going on... ##

In [ ]:
# update for new results here
new_mdn_dir = '/Users/smericks/Desktop/StrongLensing/project3/sbi-stronglensing/mdn/mdn_resnet34_2026-07-09_19-56-40/'
new_nsf_dir = '/Users/smericks/Desktop/StrongLensing/project3/sbi-stronglensing/nsf/nsf_resnet34_2026-07-05_08-37-20/'

In [ ]:
truth_vals_dict_mdn, dbls_idxs, quads_idxs, fasttdc_fpd_samps_dbls_mdn, fasttdc_fpd_samps_quads_mdn = truth_and_posteriors(
    (new_mdn_dir+'1ktest_seed7_5ksamps.npy'),
    (new_nsf_dir+'test10k_seed7.h5'), # storing test set in one place (NSF dir)
    chosen_idxs=np.arange(0,1000,1),GROUNDTRUTH_COSMO=GROUNDTRUTH_COSMO)

truth_fpd01_mdn_dbls = truth_vals_dict_mdn['fpd_01'][dbls_idxs]
fpd01_samples_mdn_dbls = fasttdc_fpd_samps_dbls_mdn[:,:,0]

truth_fpd01_quads_mdn = truth_vals_dict_mdn['fpd_01'][quads_idxs]
fpd01_samples_quads_mdn = fasttdc_fpd_samps_quads_mdn[:,:,0]

In [ ]:
truth_fpd01_mdn = np.concatenate((truth_fpd01_mdn_dbls,truth_fpd01_quads_mdn))
fpd01_samples_mdn = np.concatenate((fpd01_samples_mdn_dbls,fpd01_samples_quads_mdn),axis=0)

In [ ]:
truth_vals_dict_nsf, dbls_idxs, quads_idxs, fasttdc_fpd_samps_dbls_nsf, fasttdc_fpd_samps_quads_nsf = truth_and_posteriors(
    (new_nsf_dir+'1ktest_seed7_5ksamps.npy'),
    (new_nsf_dir+'test10k_seed7.h5'),
    chosen_idxs=np.arange(0,1000,1), GROUNDTRUTH_COSMO=GROUNDTRUTH_COSMO)


truth_fpd01_nsf_dbls = truth_vals_dict_nsf['fpd_01'][dbls_idxs]
fpd01_samples_nsf_dbls = fasttdc_fpd_samps_dbls_nsf[:,:,0]

truth_fpd01_nsf_quads = truth_vals_dict_nsf['fpd_01'][quads_idxs]
fpd01_samples_nsf_quads = fasttdc_fpd_samps_quads_nsf[:,:,0]

In [ ]:
truth_fpd01_nsf = np.concatenate((truth_fpd01_nsf_dbls,truth_fpd01_nsf_quads))
fpd01_samples_nsf = np.concatenate((fpd01_samples_nsf_dbls,fpd01_samples_nsf_quads),axis=0)

In [ ]:
fig,axs = plt.subplots(18,5,figsize=(20,55),dpi=400)
for j in range(0,18):
    lens_indices = np.arange(j*5,j*5+5)
    for i in range(0,5):
        #try:
        _,bins,_ = axs[j,i].hist(fpd01_samples_mdn[lens_indices[i]],color=MDN_COLOR,label='MDN')
        axs[j,i].hist(fpd01_samples_nsf[lens_indices[i]],color=NSF_COLOR,bins=bins,label='NSF')
        axs[j,i].vlines(truth_fpd01_nsf[lens_indices[i]],ymin=0.,ymax=1500.,color='black',zorder=200,linewidth=2)
        axs[j,i].set_title('Lens %d'%(lens_indices[i]))
        if lens_indices[i] == 0:
            axs[j,i].legend()
        #except:
        #    pass

plt.suptitle('Doubles FPD01', y=0.995,fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
fig,axs = plt.subplots(1,2,figsize=(10,5))
axs[0].scatter(truth_fpd01_mdn,np.median(fpd01_samples_mdn,axis=1),color=MDN_COLOR,label='MDN',s=10)
axs[0].plot(truth_fpd01_mdn,truth_fpd01_mdn,color='black')
axs[0].legend()
axs[0].set_title('fpd01')
axs[0].set_xlabel('truth')
axs[0].set_ylabel('pred.')
axs[1].scatter(truth_fpd01_nsf,np.median(fpd01_samples_nsf,axis=1),color=NSF_COLOR,label='NSF',s=10)
axs[1].plot(truth_fpd01_nsf,truth_fpd01_nsf,color='black')
axs[1].legend()
axs[1].set_title('fpd01')
axs[1].set_xlabel('truth')
axs[1].set_ylabel('pred.')

axs[0].vlines(-2,ymin=-2.,ymax=0.05, color='grey')
axs[0].vlines(0.05,ymin=-2.,ymax=0.05, color='grey')
axs[0].hlines(-2,xmin=-2.,xmax=0.05, color='grey')
axs[0].hlines(0.05,xmin=-2.,xmax=0.05, color='grey')

axs[1].vlines(-2,ymin=-2.,ymax=0.05, color='grey')
axs[1].vlines(0.05,ymin=-2.,ymax=0.05, color='grey')
axs[1].hlines(-2,xmin=-2.,xmax=0.05, color='grey')
axs[1].hlines(0.05,xmin=-2.,xmax=0.05, color='grey')

plt.tight_layout()

plt.figure()
fig,axs = plt.subplots(1,2,figsize=(10,5))
axs[0].scatter(truth_fpd01_mdn,np.median(fpd01_samples_mdn,axis=1),color=MDN_COLOR,label='MDN',s=10)
axs[0].plot(truth_fpd01_mdn,truth_fpd01_mdn,color='black')
axs[0].legend()
axs[0].set_title('fpd01')
axs[0].set_xlabel('truth')
axs[0].set_ylabel('pred.')
axs[1].scatter(truth_fpd01_nsf,np.median(fpd01_samples_nsf,axis=1),color=NSF_COLOR,label='NSF',s=10)
axs[1].plot(truth_fpd01_nsf,truth_fpd01_nsf,color='black')
axs[1].legend()
axs[1].set_title('fpd01')
axs[1].set_xlabel('truth')
axs[1].set_ylabel('pred.')
plt.tight_layout()

axs[0].set_xlim([-2,0.05])
axs[0].set_ylim([-2,0.05])
axs[1].set_xlim([-2,0.05])
axs[1].set_ylim([-2,0.05])

In [ ]:
sys.path.insert(0, os.path.join(dirname, '../../../sbi-stronglensing'))
from src.analysis.visualization_utils import plot_corner, plot_corner_overlay, plot_corner_image, plot_parity, table_of_metrics, plot_coverage
from src.analysis.diagnostic_utils import get_coverage

print('MDN')
table_of_metrics(np.expand_dims(np.transpose(np.abs(fpd01_samples_mdn),axes=(1,0)),axis=2),
    np.expand_dims(np.abs(truth_fpd01_mdn),axis=1),param_labels=['fpd01'])
print(' ')
print('NSF')
table_of_metrics(np.expand_dims(np.transpose(np.abs(fpd01_samples_nsf),axes=(1,0)),axis=2),
    np.expand_dims(np.abs(truth_fpd01_nsf),axis=1),param_labels=['fpd01'])

ecp_mdn, alpha_mdn, ecp_error_mdn = get_coverage(np.expand_dims(np.transpose(fpd01_samples_mdn,axes=(1,0)),axis=2), 
    np.expand_dims(truth_fpd01_mdn,axis=1), references = "random", metric = "euclidean", 
    norm = True, bootstrap=True, sigma=1)

ecp_nsf, alpha_nsf, ecp_error_nsf = get_coverage(np.expand_dims(np.transpose(fpd01_samples_nsf,axes=(1,0)),axis=2), 
    np.expand_dims(truth_fpd01_nsf,axis=1), references = "random", metric = "euclidean", 
    norm = True, bootstrap=True, sigma=1)

ecp_list = [ecp_mdn, ecp_nsf]
alpha_list = [alpha_mdn, alpha_nsf]
ecp_error_list = [ecp_error_mdn, ecp_error_nsf]
colors = [MDN_COLOR, NSF_COLOR]
legend_labels = ['MDN', 'NSF']
plot_coverage(alpha_list, ecp_list, ecp_error_list, colors, legend_labels, quality_metrics=['ECE', 'AUC'], 
    title='$\Delta \phi_{01}$')
plt.savefig('/Users/smericks/Desktop/StrongLensing/project3/Figures/fpd01_coverage.pdf',bbox_inches='tight')

## Curating realistic test set w/ selection cuts ##

In [ ]:
max_td_days = np.zeros(len(truth_vals_dict_nsf['deflector_gamma_pl']))
max_td_days[dbls_idxs] = np.abs(truth_vals_dict_nsf['td_01'][dbls_idxs])
max_td_days[quads_idxs] = np.max(np.vstack( # vstack creates shape=(3,1000)
    (np.abs(truth_vals_dict_nsf['td_01'][quads_idxs]),
    np.abs(truth_vals_dict_nsf['td_02'][quads_idxs]),
    np.abs(truth_vals_dict_nsf['td_03'][quads_idxs]))),axis=0)

In [ ]:
new_file = False
if new_file:

    # new way of selecting for a narrower, Gaussian distribution in gamma, and td > 10 days
    td_tenplus_idx = np.where(
        (max_td_days > 10.) & 
            (max_td_days < 150.)
    )[0]

    gamma_tenplus = truth_vals_dict_nsf['deflector_gamma_pl'][td_tenplus_idx]


    # resample based on gamma_tenplus
    import numpy as np

    # Your original data
    x = gamma_tenplus

    # Define narrow Gaussian probability weights
    mu_narrow = 2.
    sigma_narrow = 0.1  # Make it narrower (adjust factor as needed)

    # Calculate sampling probabilities
    weights = np.exp(-0.5 * ((x - mu_narrow) / sigma_narrow)**2)
    weights = weights / weights.sum()  # Normalize

    # Sample 100 indices
    selected_indices = np.random.choice(len(x), size=200, replace=False, p=weights)

    narrow_gamma_td_tenplus_idx = td_tenplus_idx[selected_indices]

    print(selected_indices)
    print(np.sort(narrow_gamma_td_tenplus_idx))

else:
    narrow_gamma_td_tenplus_idx = [ # gamma centered ~ 2.03, time-delay > 10 days 
        3, 7, 12, 18, 22, 34, 35, 37, 41, 80, 82, 84, 91, 93, 109, 113, 114, 122,
        128, 129, 139, 148, 154, 157, 164, 166, 167, 170, 173, 179, 191, 193, 196,
        209, 212, 220, 246, 251, 254, 255, 260, 264, 269, 271, 272, 274, 275, 278,
        285, 288, 290, 292, 293, 297, 299, 300, 301, 304, 312, 314, 317, 320, 321,
        328, 329, 332, 333, 334, 339, 340, 341, 342, 346, 352, 357, 359, 360, 373,
        375, 393, 397, 408, 409, 414, 416, 423, 426, 433, 436, 437, 439, 440, 445,
        453, 462, 465, 472, 477, 479, 481, 487, 501, 507, 509, 510, 511, 515, 520,
        522, 523, 529, 535, 536, 539, 540, 543, 550, 568, 570, 572, 576, 579, 583,
        606, 610, 619, 620, 628, 629, 633, 638, 642, 656, 658, 661, 673, 676, 683,
        689, 693, 695, 699, 713, 719, 723, 728, 731, 737, 738, 739, 748, 750, 753,
        756, 766, 780, 784, 789, 793, 797, 799, 811, 815, 816, 821, 824, 829, 843,
        855, 857, 863, 865, 867, 872, 886, 889, 900, 903, 908, 913, 916, 921, 927,
        938, 940, 950, 955, 956, 959, 964, 966, 970, 972, 974, 976, 977, 978, 985,
        988, 991
    ]

In [ ]:
plt.hist(truth_vals_dict_nsf['deflector_gamma_pl'][narrow_gamma_td_tenplus_idx])
print(np.mean(truth_vals_dict_nsf['deflector_gamma_pl'][narrow_gamma_td_tenplus_idx]))
print(np.std(truth_vals_dict_nsf['deflector_gamma_pl'][narrow_gamma_td_tenplus_idx]))


Checking for signs of distribution shift...

In [ ]:
all_test_points = np.stack(
    (np.exp(truth_vals_dict_nsf['deflector_LOG_theta_E']),
     truth_vals_dict_nsf['deflector_gamma_pl'])).T

fig = corner.corner(data=all_test_points,color='grey',fig=None,
                    plot_datapoints=False,fill_contours=True,levels=[0.68,0.95],
                    labels=['theta_E','gamma_pl'],dpi=300,smooth=2,
                    hist_kwargs={'density':True})

corner.corner(data=all_test_points[narrow_gamma_td_tenplus_idx,:],color=NSF_COLOR,
            fig=fig,
            plot_datapoints=False,fill_contours=True,levels=[0.68,0.95],
            labels=['theta_E','gamma_pl'],dpi=300,smooth=2,
            hist_kwargs={'density':True})